## Setup

In [1]:
import sys
import os
import importlib

from tabulate import tabulate
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

sys.path.append("../src")

import utils
import plot
import rstats
import mappings

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(plot)
importlib.reload(rstats)
importlib.reload(mappings)

# NOTE: define variables in .env to avoid changing the notebook
load_dotenv("../.env", override=True)

True

In [2]:
FILENAME = os.getenv("FILENAME", "UFABC_PLT_combined")
BACKEND = os.getenv("BACKEND", "gemini")
DST = f"../results/{BACKEND}/{FILENAME}/"

print(f"{FILENAME=}")
print(f"{BACKEND=}")

df = utils.load(f"../results/{BACKEND}/{FILENAME}/metrics.csv")
print(f"{df.shape=}")

# Avoid log(0) issues
eps = 1e-10
df.loc[df.entropy <= 0, "entropy"] = eps
df.loc[df.d_next <= 0, "d_next"] = eps
df.loc[df.vel <= 0, "vel"] = eps
df.loc[df.acc <= 0, "acc"] = eps

FILENAME='german'
BACKEND='openai'
df.shape=(10010, 11)


In [3]:
tmp = ["id", "category", "concept"] if "category" in df.columns else ["id", "concept"]
grouped = df.groupby(tmp, as_index=False)
dfx = grouped.mean(numeric_only=True)
print(tabulate(dfx.head(3), headers="keys", showindex=False))

id    category    concept      num     d_next    entropy    d_centroid       vel       acc
----  ----------  ---------  -----  ---------  ---------  ------------  --------  --------
s1    bird        Eule        47.5  0.155163         nan      0.267218  0.530535  0.821104
s1    bird        Möwe       110.5  0.219002         nan      0.297917  0.638331  1.06533
s1    bird        Specht      84.5  0.0649124        nan      0.281603  0.345953  0.500833


## Analysis

In [ ]:
import io
import contextlib

metrics = ["d_next", "vel", "acc", "entropy", "d_centroid"]
if "category" not in dfx.columns:
    dfx = dfx.rename(columns={"concept": "category"})

dfx["category"] = dfx["category"].map(mappings.categories.get(FILENAME))
cats = sorted(pd.Series(dfx["category"].unique()).tolist())

gridspec_kw = {"height_ratios": [1, 1], "hspace": 0.35}
figsize = mappings.figsize.get(FILENAME)
fig, axes = plt.subplots(2, len(metrics), figsize=figsize, gridspec_kw=gridspec_kw)

for i, metric in enumerate(metrics):
    print(f"[{i + 1}/{len(metrics)}] Analyzing '{metric}'")

    # Use lognormal for all metrics, except for d_centroid
    family = "lognormal" if metric != "d_centroid" else "gaussian"

    stdout = io.StringIO()
    with contextlib.redirect_stdout(stdout):
        # Fit GLMM and get emmeans + Tukey pairs
        glmm = rstats.glmm(dfx, formula=f"{metric} ~ category + (1|id)", family=family)
        pred = rstats.emmeans(effect="category")
        pairs = rstats.pairs()

    # Standardize column names for EMMs and plit "A - B" contrasts into separate columns
    if "emmean" not in pred and "response" in pred:
        pred = pred.rename(columns={"response": "emmean"})
    if "SE" not in pred and "SE.df" in pred:
        pred = pred.rename(columns={"SE.df": "SE"})
    if "contrast" in pairs.columns and not {"group1", "group2"}.issubset(pairs.columns):
        pairs[["group1", "group2"]] = pairs["contrast"].str.split(" - ", expand=True)

    # Save R output
    with open(f"{DST}/r-output-{metric}.txt", "w") as fp:
        fp.write(stdout.getvalue())

    # Individual figure: boxplot
    # ax1 = plot.boxplot(dfx, metric, pred, pairs, cats=cats, ax=None, figsize=mappings.figsize.get(FILENAME))
    # ax1.set_title(mappings.ylabels.get(metric, metric))
    # plt.tight_layout()
    # plt.savefig(f"{DST}/boxplot-{metric}.png", bbox_inches="tight")
    # plt.close()

    # Combined figure: boxplot
    ax2 = plot.boxplot(dfx, metric, pred, pairs, cats=cats, ax=axes[0, i])
    ax2.set_title(mappings.ylabels.get(metric, metric))

    # Combined figure: heatmap
    hm, pmat = plot.heatmap(cats=cats, pairs=pairs, ax=axes[1, i])
    pmat.to_csv(f"{DST}/pvalues-{metric}.csv")

cax = fig.add_axes([0.25, -0.05, 0.5, 0.03])
cbar = fig.colorbar(
    hm.collections[0],
    cax=cax,
    orientation="horizontal",
    ticks=[5e-5, 5.5e-4, 5.5e-3, 3e-2, 0.55],
)
cbar.ax.set_xticklabels(["$p < 1e-4$", "$p < 1e-3$", "$p < 1e-2$", "$p < 5e-2$", "ns"])

tmp = {
    "swear-fluency": "swearwords",
    "italian": "italian",
    "german": "german",
    "parkinson": "parkinson",
}
tmp = tmp.get(FILENAME)

fig.savefig(f"{DST}/{BACKEND.lower()}-{tmp}.pdf", dpi=300, bbox_inches="tight")
plt.close()

[1/5] Analyzing 'd_next'
[2/5] Analyzing 'vel'
[3/5] Analyzing 'acc'
[4/5] Analyzing 'entropy'
[5/5] Analyzing 'd_centroid'
